[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/00_api_access.ipynb)

# Part 0 — API access

> **Run this first.** Every other notebook opens with the same setup block.

Get a free key at <https://aistudio.google.com/apikey>. In Colab, add it to the **Secrets**
panel (key icon, left sidebar) as `GEMINI_API_KEY`. Locally, `export GEMINI_API_KEY=...`

Never paste a key into a cell — it ends up in the `.ipynb`, your shell history, and git.

In [ ]:
# export GEMINI_API_KEY=xxx

In [ ]:
# Install dependencies (run once).
import sys
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0"


In [ ]:
import os, time
from google import genai
from google.genai import types as gtypes

# Load the API key. In Colab use the Secrets panel; locally use an environment variable.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
# Free-tier limits for this model, confirmed Sept 2026: 15 RPM, 250k TPM, 1000 RPD.
# Agent loops are bursty, so the per-minute cap is what the retry wrapper below absorbs;
# the daily cap is the one that ends a session.
MODEL = "gemini-3.1-flash-lite"

In [ ]:
def generate_with_retry(*, contents, config=None, max_attempts=6):
    """client.models.generate_content with exponential backoff on 429.

    Free-tier Gemini caps requests/minute. A ReAct loop can fire many calls
    back-to-back and trip the limit; we sleep and retry instead of crashing.
    """
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)

print(f"Gemini client ready (model={MODEL}).")

In [ ]:
# Smoke test: one chat call, no tools.
resp = generate_with_retry(
    contents="Say hello in one short sentence.",
)
print(resp.text)

## Available models

`gemini-3.1-flash-lite` is the default here because it has the most generous free-tier
limits. Agent loops are bursty, so the daily request cap matters more than raw quality.

To use a different one, change `MODEL` above. The cell below lists what your key can
actually reach — rather than a hard-coded list that goes stale.

In [ ]:
for m in client.models.list():
    acts = m.supported_actions or []
    if acts and "generateContent" not in acts:
        continue
    print(f"{m.name:42} in={m.input_token_limit or '?':>9} out={m.output_token_limit or '?':>7}")

## Using OpenAI or Anthropic instead

> **Untested.** Everything in these notebooks was written and run against Gemini. The
> snippets below are correct as far as each provider's documented API goes, but we have
> not run the workshop material through them. If you use one and something breaks, that
> is a bug in our material, not in yours — tell us.

You need to change two things: how the client is built, and how tool calls are spelled.
Everything else — the loops, the graphs, the exemplars — is provider-agnostic.

### Anthropic

```python
%pip install -U -q anthropic
import anthropic

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

resp = client.messages.create(
    model="claude-sonnet-5",          # also: claude-opus-5, claude-haiku-4-5
    max_tokens=1024,                  # required
    system="You are terse.",
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
)
print(resp.content[0].text)

# what your key can reach:
for m in client.models.list():
    print(m.id)
```

### OpenAI

```python
%pip install -U -q openai
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

resp = client.chat.completions.create(
    model="...",                      # pick one from client.models.list()
    messages=[{"role": "system", "content": "You are terse."},
              {"role": "user",   "content": "Say hello in one short sentence."}],
)
print(resp.choices[0].message.content)

# what your key can reach:
for m in client.models.list():
    print(m.id)
```

### The three things that differ

|                | Gemini                          | Anthropic                                      | OpenAI                                    |
|----------------|---------------------------------|------------------------------------------------|-------------------------------------------|
| tool schema    | `FunctionDeclaration(...)`      | `{"name", "description", "input_schema"}`       | `{"type": "function", "function": {...}}`  |
| model asks     | `part.function_call`            | content block with `type="tool_use"`            | `message.tool_calls[]`                     |
| you answer     | `Part.from_function_response`   | `{"type": "tool_result", "tool_use_id": ...}`   | `{"role": "tool", "tool_call_id": ...}`    |

Same three ideas, three spellings. That similarity is why a thin shim works — and it is
the gap MCP closes at the transport layer.